# Python编程基础 · 工单1.2《工业数据文件处理系统》 — 教师参考版
第8周 | 参考实现可直接运行验证 | 答案请勿下发学生

In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


工作目录: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/02-程序设计基础


## 参考实现 `w08`（含依赖导入）

In [2]:
import json
import random
import re
import sys
import numpy as np  # 通用科学计算


def w08():
    import csv
    import os
    random.seed(7)
    raw = "raw_demo.csv"
    if not os.path.exists(raw):
        with open(raw, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["device_id", "temp", "press", "status"])
            for i in range(60):
                t = round(random.uniform(40, 90), 1)
                if i % 17 == 3:
                    t = ""
                if i % 23 == 5:
                    t = "err"
                if i % 29 == 7:
                    t = 135.0
                w.writerow([f"DEV-{i % 3 + 1:02d}", t,
                            round(random.uniform(.3, .6), 2), "RUN"])

    def validate(row):
        t = row["temp"]
        if t in ("", None):
            return "缺失"
        try:
            t = float(t)
        except ValueError:
            return "类型"
        if not 0 <= t <= 120:
            return "范围"
        return None

    ok, bad, counter = 0, [], {"缺失": 0, "类型": 0, "范围": 0}
    with open(raw, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            err = validate(row)
            if err is None:
                ok += 1
            else:
                bad.append((row, err))
                counter[err] += 1
    with open("bad_records.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["device_id", "temp", "err"])
        for row, e in bad:
            w.writerow([row["device_id"], row["temp"], e])
    print(f"清洗报告: 总60 成功{ok} 问题{len(bad)} {counter}")


## 学生版TODO → 教师答案对照
学生版starter中的TODO与上方参考实现（或下方双解）的对应关系：

| TODO | 题面（学生看到的指导） |
|------|----------------------|
| `?` | （在此处编写你的代码——依据上方注释/文档字符串） |
| `2` | TODO-2: with open读取RAW, 逐行validate |
| `3` | TODO-3: bad_rows写入BAD(带表头) TODO-4: 输出清洗报告: 总数/成功/各错误类型计数 |

> 发放纪律：本对照表**勿随学生版下发**；课堂按层放提示（D层演示照做/C层只给思路行）。

## 多解对比（已实跑互验）
| 解 | 路线 | 结论 |
|----|------|------|
| A | csv逐行+validate | 教学显式（缺失/类型/范围三分类） |
| B | pandas一步过滤 | 2留3弃一致，代码量1/3 |

> 决策素材：清洗报告要分类统计→A；只要结果→B。

## 执行与验证

In [3]:
w08()

清洗报告: 总60 成功51 问题9 {'缺失': 4, '类型': 3, '范围': 2}
